# Initialisation
Code below is for preppring tools for use

---

In [47]:
# import plotting tools
# plotnine is the python clone of ggplot2 from R
# this should be used over plt where possible

import pandas as pd
import matplotlib as plt
from plotnine import *
import numpy as np

In [48]:
from collections import *
from skimage import *
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [49]:
from transformers import AutoTokenizer, pipeline, AutoModel
import torch
import re
from thefuzz import process

# Testing
The code below is for the testing of basic premise and ideas, such as probing raw statistics and the like. This may also be used for getting familuar with certain tools and datasets.

In [50]:
df = pd.read_csv('att_sch_stats/att_sch.csv')

attainment_2025 = df[df['DateCode'].astype(str).str.contains("2024-25", na=False)]
above_5_2025 = attainment_2025[
    (
        attainment_2025['Number Of Awards'].astype(str).str.contains("5", na=False)
    ) & (
        attainment_2025['SCQF Level'].astype(str).str.contains("6", na=False)
    )
]
att_list = above_5_2025[["GeographyName", "Value"]]
att_list = att_list.sort_values(by="GeographyName")
att_list = att_list[att_list.GeographyName != "National"]

supressors = ["*", "#"]

supressed_schools = att_list[
    att_list["Value"].astype(str).isin(supressors)
].sort_values(by="GeographyName")

att_list = att_list[
    ~att_list["Value"].astype(str).isin(supressors)
].copy()

att_list

,GeographyName,Value
164471,St Joseph's College,47
155091,Aberdeen Grammar School,72
155511,Aboyne Academy,46
171891,Airdrie Academy,35
154111,Alford Academy,41
...,...,...
171261,Whitehill Secondary School,22
151241,Wick High School,18
174131,Williamwood High School,82
174061,Woodfarm High School,74


In [51]:
primary_ys = ["P1", "P2", "P3", "P4", "P5", "P6", "P7"]

sch_sum_2025 = pd.read_excel('sch_sum_stats/sch_sum_2025.xlsx', sheet_name=2, header=1)
hs_sum_2025 = sch_sum_2025[sch_sum_2025["School Type"].astype(str).str.contains("Secondary")]
hs_sum_2025 = hs_sum_2025.drop(columns=primary_ys).drop(columns="Special school pupils")
hs_sum_2025 = hs_sum_2025.sort_values(by="School Name")
hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).isin(["Braeview Academy", "Craigie High School", "Glasgow Gaelic School", "St Joseph's Academy", "The Community School of Auchterarder"])]

supressed_names = supressed_schools["GeographyName"].to_list()
pattern = '|'.join(re.escape(name) for name in supressed_names)
hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).str.contains(pattern, case=False)].copy()

# hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).isin(supressed_schools["GeographyName"])].copy()
# hs_sum_2025 = hs_sum_2025[hs_sum_2025["School Name"].astype(str).isin(att_list["GeographyName"])]
att_list = att_list[att_list["GeographyName"].astype(str).isin(hs_sum_2025["School Name"].astype(str))].copy()
hs_sum_2025 = hs_sum_2025[hs_sum_2025["School Name"].astype(str).isin(att_list["GeographyName"].astype(str))].copy()

In [52]:
target = "Value"

simd_columns = [
    "SIMD Quintile 1 [Note 6]",
    "SIMD Quintile 2 [Note 6]",
    "SIMD Quintile 3 [Note 6]",
    "SIMD Quintile 4 [Note 6]",
    "SIMD Quintile 5 [Note 6]"
]

numeric_columns = simd_columns + ["Pupil roll"]

for column in numeric_columns:
    hs_sum_2025[column] = pd.to_numeric(
        hs_sum_2025[column]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

for column in simd_columns:
    hs_sum_2025[f"{column} proportion"] = (
        hs_sum_2025[column] / hs_sum_2025["Pupil roll"]
    )

simd_features = [
    f"{column} proportion"
    for column in simd_columns
]


In [53]:
URC = ["Urban [Note 7]", "Small town [Note 7]", "Rural [Note 7]"]

def find_midpoint(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip().replace("%","")
    lower, upper = val.split("-")
    return (float(lower) + float(upper)) / 200

for column in URC:
    hs_sum_2025[f"{column} proportion"] = (
        hs_sum_2025[column].apply(find_midpoint)
    )

URF = [
    f"{column} proportion"
    for column in URC
]

In [54]:
denomination = "Denomination of school"

hs_sum_2025["Denomination Encoded"] = (
    hs_sum_2025[denomination]
    .map({
        "Non-denominational school" : 0,
        "Roman Catholic School" : 1
    })
)

In [55]:
features = [
    "Average class size",
    "Pupil roll",
    "Pupil Teacher Ratio",
    "Pupils with an Additional Support Need recorded",
    "Denomination Encoded",
    *simd_features,
    *URF,
]

In [56]:
data = hs_sum_2025.merge(
    att_list[["GeographyName", "Value"]],
    left_on="School Name",
    right_on="GeographyName",
    how="inner",
    validate="one_to_one"
)

x = data[features]
y = pd.to_numeric(data["Value"], errors="coerce")

valid = y.notna()
x = x.loc[valid].copy()
y = y.loc[valid].copy()
numeric_features = features

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features)
])

In [57]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

In [58]:
model = Pipeline([
    ("Preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=500,
        random_state=42,
    ))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 8.33334328358209
RMSE: 10.665828484169964
R²: 0.46588860382718267


In [59]:
# import matplotlib.pyplot as plt

# # 1. Calculate average Pupil Teacher Ratio per council
# avg_ptr = db.groupby("Local Authority")["Pupil Teacher Ratio"].mean().reset_index()

# # 2. Create figure and axes object properly
# fig, ax = plt.subplots(figsize=(10, 6))

# # 3. Plot a horizontal bar chart
# ax.barh(avg_ptr["Local Authority"], avg_ptr["Pupil Teacher Ratio"], color="skyblue")

# ax.set_xlabel("Average Pupil Teacher Ratio")
# ax.set_ylabel("Local Authority")
# ax.set_title("Average Pupil Teacher Ratio by Council")

# plt.tight_layout()
# plt.show()
print(hs_sum_2025[numeric_columns].isna().sum())

SIMD Quintile 1 [Note 6]    32
SIMD Quintile 2 [Note 6]    23
SIMD Quintile 3 [Note 6]     7
SIMD Quintile 4 [Note 6]    19
SIMD Quintile 5 [Note 6]    34
Pupil roll                   0
dtype: int64


---

# Testing previous years
starting 2024
Code may be duplicated before possible functions are identified

In [60]:
df = pd.read_csv('att_sch_stats/att_sch.csv')

attainment_2024 = df[df['DateCode'].astype(str).str.contains("2023-24", na=False)]
above_5_2024 = attainment_2024[
    (
        attainment_2024['Number Of Awards'].astype(str).str.contains("5", na=False)
    ) & (
        attainment_2024['SCQF Level'].astype(str).str.contains("6", na=False)
    )
]
att_list_24 = above_5_2024[["GeographyName", "Value"]]
att_list_24 = att_list_24.sort_values(by="GeographyName")
att_list_24 = att_list_24[att_list_24.GeographyName != "National"]

supressors = ["*", "#"]

supressed_schools = att_list_24[
    att_list_24["Value"].astype(str).isin(supressors)
].sort_values(by="GeographyName")

att_list_24 = att_list_24[
    ~att_list_24["Value"].astype(str).isin(supressors)
].copy()

att_list_24= att_list_24.dropna()

In [61]:
primary_ys = ["P1", "P2", "P3", "P4", "P5", "P6", "P7"]

sch_sum_2024 = pd.read_excel('sch_sum_stats/sch_sum_2024.xlsx', sheet_name=2, header=1)
hs_sum_2024 = sch_sum_2024[sch_sum_2024["School Type"].astype(str).str.contains("Secondary")]
hs_sum_2024 = hs_sum_2024.drop(columns=primary_ys).drop(columns="Special school pupils")
hs_sum_2024 = hs_sum_2024.sort_values(by="School Name")
# hs_sum_2024 = hs_sum_2024[~hs_sum_2024["School Name"].astype(str).isin(["Braeview Academy", "Craigie High School", "Glasgow Gaelic School", "St Joseph's Academy", "The Community School of Auchterarder"])]

supressed_names = supressed_schools["GeographyName"].to_list()
pattern = '|'.join(re.escape(name) for name in supressed_names)
hs_sum_2024 = hs_sum_2024[~hs_sum_2024["School Name"].astype(str).str.contains(pattern, case=False)].copy()

# hs_sum_2025 = hs_sum_2025[~hs_sum_2025["School Name"].astype(str).isin(supressed_schools["GeographyName"])].copy()
# hs_sum_2025 = hs_sum_2025[hs_sum_2025["School Name"].astype(str).isin(att_list["GeographyName"])]
att_list_24 = att_list_24[att_list_24["GeographyName"].astype(str).isin(hs_sum_2024["School Name"].astype(str))].copy()
hs_sum_2024 = hs_sum_2024[hs_sum_2024["School Name"].astype(str).isin(att_list_24["GeographyName"].astype(str))].copy()

hs_sum_2024

,Local Authority,SeedCode,School Name,School Type,Teachers (Full Time Equivalent) [Note 1],Pupil roll,Pupil Teacher Ratio,Average class size,Female,Male,...,Small town [Note 7],Rural [Note 7],Urban/rural classification not available [Note 7],Attendance rate (%)[Note 8] [Note 9],Authorised absence rate (%) [Note 8] [Note 9],Unauthorised absence rate (%)[Note 8] [Note 9],Absence due to temporary exclusion rate (%) [Note 8] [Note 9],School's 6-fold urban/rural classification,School's 8-fold urban/rural classification,Denomination of school
6,Aberdeen City,5244439,Aberdeen Grammar School,Secondary,101.2,1310,12.9,0,628,682,...,0-10%,0-10%,0-10%,92.66,5.81,1.5,0.03,Large urban areas,Large urban areas,Non-denominational
19,Aberdeenshire,5257530,Aboyne Academy,Secondary,59.6,659,11.1,0,315,344,...,0-10%,90-100%,0-10%,90.94,6.64,2.41,C,Remote rural areas,Remote rural areas,Non-denominational
27,North Lanarkshire,8505438,Airdrie Academy,Secondary,74.7,1044,14.0,0,533,511,...,0-10%,20-30%,0-10%,83.47,6.18,10.32,0.03,Other urban areas,Other urban areas,Non-denominational
39,Aberdeenshire,5230934,Alford Academy,Secondary,60.6,687,11.3,0,316,371,...,0-10%,90-100%,0-10%,89.83,6.64,3.46,0.06,Remote rural areas,Remote rural areas,Non-denominational
43,Glasgow City,8431930,All Saints Secondary School,Secondary,67.2,911,13.6,0,474,437,...,0-10%,0-10%,0-10%,86.93,7.06,5.99,0.02,Large urban areas,Large urban areas,Roman Catholic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2402,Glasgow City,8439532,Whitehill Secondary School,Secondary,61.4,618,10.1,0,287,331,...,0-10%,0-10%,0-10%,83.37,4.49,12.13,C,Large urban areas,Large urban areas,Non-denominational
2411,Highland,5102138,Wick High School,Secondary,56.5,736,13.0,0,370,366,...,50-60%,40-50%,0-10%,85.7,8.67,5.62,C,Remote small towns,Very Remote Small Towns,Non-denominational
2415,East Renfrewshire,8602034,Williamwood High School,Secondary,140.8,1678,11.9,0,843,835,...,0-10%,0-10%,0-10%,91.95,5.39,2.65,0.01,Large urban areas,Large urban areas,Non-denominational
2428,East Renfrewshire,8601836,Woodfarm High School,Secondary,74.8,701,9.4,0,333,368,...,0-10%,0-10%,0-10%,91.68,6.19,2.11,0.01,Large urban areas,Large urban areas,Non-denominational


In [62]:
# School name columns
att_schools = att_list_24["GeographyName"]
hs_schools = hs_sum_2024["School Name"]

# Schools in att_list but not hs_sum_2024
missing_from_hs = sorted(set(att_schools) - set(hs_schools))

# Schools in hs_sum_2024 but not att_list
missing_from_att = sorted(set(hs_schools) - set(att_schools))

differences = pd.DataFrame({
    "School Name": (
        [*missing_from_hs, *missing_from_att]
    ),
    "Missing From": (
        ["hs_sum_2024"] * len(missing_from_hs)
        + ["att_list"] * len(missing_from_att)
    )
})

display(differences)

,School Name,Missing From


In [63]:
att_list_24

,GeographyName,Value
129890,Aberdeen Grammar School,74
130310,Aboyne Academy,47
146691,Airdrie Academy,30
128910,Alford Academy,39
144311,All Saints Secondary School,27
...,...,...
146061,Whitehill Secondary School,17
126040,Wick High School,23
148931,Williamwood High School,77
148861,Woodfarm High School,83


In [64]:
target = "Value"

simd_columns = [
    "SIMD Quintile 1 [Note 6]",
    "SIMD Quintile 2 [Note 6]",
    "SIMD Quintile 3 [Note 6]",
    "SIMD Quintile 4 [Note 6]",
    "SIMD Quintile 5 [Note 6]"
]

numeric_columns = simd_columns + ["Pupil roll"]

for column in numeric_columns:
    hs_sum_2024[column] = pd.to_numeric(
        hs_sum_2024[column]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

for column in simd_columns:
    hs_sum_2024[f"{column} proportion"] = (
        hs_sum_2024[column] / hs_sum_2024["Pupil roll"]
    )

simd_features = [
    f"{column} proportion"
    for column in simd_columns
]


In [65]:
URC = ["Urban [Note 7]", "Small town [Note 7]", "Rural [Note 7]"]

for column in URC:
    hs_sum_2024[f"{column} proportion"] = (
        hs_sum_2024[column].apply(find_midpoint)
    )

URF = [
    f"{column} proportion"
    for column in URC
]

In [66]:
denomination = "Denomination of school"

hs_sum_2024["Denomination Encoded"] = (
    hs_sum_2024[denomination]
    .map({
        "Non-denominational" : 0,
        "Roman Catholic" : 1
    })
)

In [67]:
features = [
    "Average class size",
    "Pupil roll",
    "Pupil Teacher Ratio",
    "Pupils with an Additional Support Need recorded",
    "Denomination Encoded",
    *simd_features,
    *URF,
]

In [68]:
data = hs_sum_2024.merge(
    att_list_24[["GeographyName", "Value"]],
    left_on="School Name",
    right_on="GeographyName",
    how="inner",
    validate="one_to_one"
)

x = data[features]
y = pd.to_numeric(data["Value"], errors="coerce")

valid = y.notna()
x = x.loc[valid].copy()
y = y.loc[valid].copy()
numeric_features = features

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features)
])

In [69]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

In [70]:
model = Pipeline([
    ("Preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=500,
        random_state=42,
    ))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 8.338676470588235
RMSE: 10.490571797907815
R²: 0.48083708354358057
